# 02 · Broad basketball feature research

**3,106 candidates; training-only screening; matched temporal ablations.** This is a deliberately broad hypothesis search, not a claim that every generated column is useful. Every displayed result is loaded from a completed, checksum-verified run of the code below. Review mode reads published evidence; train mode actually rebuilds and fits notebook 02.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display
from march_mania.notebook_support import table, style
from march_mania.publication.workflow import evidence, execution_mode, feature_stage, require_recorded_source
from march_mania.advanced_features import candidate_blocks
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file()
MODE = execution_mode()  # Set "train" to build or resume the current feature experiment.
style()
if MODE == "train":
    feature_stage(ROOT)
RESULTS, RECORD = evidence(ROOT, "feature_store")
require_recorded_source(ROOT, "feature_store", RECORD)
SUMMARY = RECORD["summary"]
assert SUMMARY["feature_count"] == len(candidate_blocks()["full"])
assert SUMMARY["status"] == "completed"
display(Markdown(f"**Verified feature run:** `{SUMMARY['fingerprint']}`  \n"
                 f"**Candidates:** {SUMMARY['feature_count']:,} · "
                 f"**Temporal ablation fits:** {SUMMARY['fold_tasks']:,}"))


In [ ]:
# One notebook output with interactive Plotly and a static PNG fallback for GitHub.
import base64
import io
import matplotlib.pyplot as plt

def bar_view(frame, x, y, title):
    data = frame.sort_values(y).copy()
    fig = px.bar(data, x=y, y=x, orientation="h", title=title,
                 hover_data=list(data.columns), labels={"macro_season_brier": "Mean season Brier · lower is better"}, height=max(350, 26 * len(data) + 100))
    fig.update_layout(template="plotly_white", margin=dict(l=150, r=25, t=65, b=55))
    fig.update_yaxes(autorange="reversed")
    static, ax = plt.subplots(figsize=(9.5, max(3.2, .27 * len(data) + 1.2)))
    ax.barh(data[x].astype(str), data[y], color="#147d82")
    ax.invert_yaxis()
    ax.set(xlabel="Mean season Brier · lower is better" if y == "macro_season_brier" else y.replace("_", " "), title=title)
    ax.spines[["top", "right"]].set_visible(False)
    static.tight_layout()
    buffer = io.BytesIO()
    static.savefig(buffer, format="png", dpi=140)
    plt.close(static)
    display({"application/vnd.plotly.v1+json": json.loads(fig.to_json()),
             "image/png": base64.b64encode(buffer.getvalue()).decode()}, raw=True)


## The candidate space

The existing 124 signals are supplemented by 2,944 distribution, venue, opponent-strength, trajectory and peer-profile candidates, plus 26 official coach-history and 12 conference-strength signals. Fixed windows cover the season, last seven games, last 30 days and last 60 days. Summaries cover center, spread, tails, skew and support. Rates use basketball denominators; undefined rates stay missing. Candidate counts are not independent statistical hypotheses: correlation screening deliberately removes redundant representations.

In [ ]:
registry = pd.read_csv(RESULTS / "feature_registry.csv")
assert registry.feature.is_unique and len(registry) == 3106
assert not {"y", "ID", "Season", "DayNum"}.intersection(registry.feature)
inventory = registry.groupby("family").agg(candidates=("feature", "size"), provenance=("source", "first")).reset_index()
table(inventory)
bar_view(inventory, "family", "candidates", "Candidate allocation by basketball mechanism")

## Availability is checked season by season

All current-season performance is frozen at day 132. Coach identity must be active at that cutoff; coach performance and tenure use strictly earlier seasons. Missing women’s coach records are not invented. Men’s Massey coverage is measured from legal publication dates for every modeled season, rather than assuming rankings exist only in recent years. Women’s models use the common/no-Massey schema.

In [ ]:
coverage = pd.read_csv(RESULTS / "coverage.csv")
table(coverage, {"detailed_coverage": "{:.1%}", "clean_coverage": "{:.1%}"})
display(Markdown("**Verified source availability:** `" + json.dumps(SUMMARY["sources"], sort_keys=True) + "`"))
assert not coverage.query("Gender == 'M' and Season in [2016,2017,2018,2019,2021]").ranking_teams.eq(0).any()
source_coverage = pd.read_csv(RESULTS / "source_coverage.csv")
table(source_coverage.query("Gender == 'M'")[["Season", "regular_teams", "ranking_teams", "ranking_systems", "ranking_fraction", "coach_teams", "conference_teams", "latest_ranking_day"]])
assert source_coverage.latest_ranking_day.dropna().le(132).all()

# Check training and inference coverage as well as the five outer validation seasons.
modeled_men = source_coverage.query("Gender == 'M' and 2013 <= Season <= 2026")
assert set(modeled_men.Season) == set(range(2013, 2027))
assert modeled_men.ranking_teams.eq(modeled_men.regular_teams).all()
assert modeled_men.coach_teams.eq(modeled_men.regular_teams).all()
assert modeled_men.latest_ranking_day.lt(132).all()
display(Markdown("**Coverage boundary verified:** every men's team has legal Massey and coach records in 2013–2026. Research fits use strictly earlier seasons starting in 2013; final fitting uses 2013–2025 excluding the canceled 2020 tournament. Massey starts in 2003 in the raw history, but no 1985–2002 tournament enters these models. Coverage of individual systems varies; absent systems are not treated as observed ranks. Women's models exclude Massey."))


## Historical targets never cross a season boundary

Team, seed and ranking-bin encodings retain each historical season’s seed and ranking. Season Y uses outcomes strictly before Y; the entire current tournament is embargoed. Fixed priors and smoothing are not estimated from future validation seasons. Coach regular-season and tournament histories also exclude current-season outcomes. Automated mutation tests alter future labels and post-cutoff games and require identical earlier features.

In [ ]:
encoding = pd.read_csv(RESULTS / "encoding_audit.csv")
observed = encoding.loc[encoding.history_max_season.notna()]
assert (observed.history_max_season < observed.Season).all()
table(encoding.groupby(["Gender", "Season"]).agg(encoding_families=("category", "size"), history_max_season=("history_max_season", "max")).reset_index())

## Screening happens inside each training fold

There is no global feature list selected using validation outcomes. Each training-only screen audits missingness, constants, rare nonzero support, univariate association and absolute correlation. A fixed capacity of 128 limits model dimensionality; duplicate and sign-reversed duplicate signals compete for one slot. Selection, imputation and fitted models are persisted together. Correlation pruning does not prove that rejected inputs are causally irrelevant, and univariate screening can miss pure interaction effects.

In [ ]:
screening = pd.read_csv(RESULTS / "screening_summary.csv")
assert (screening.candidate_count == screening.retained_count + screening.rejected_count).all()
assert screening.retained_count.between(1, 128).all()
full = screening.query("block == 'full'")
table(full[["Gender", "Season", "model", "candidate_count", "retained_count", "rejected_count"]])
reasons = [c for c in ["all_missing", "constant", "near_constant", "redundant", "capacity", "no_training_signal", "retained"] if c in full]
table(full.groupby(["Gender", "model"])[reasons].sum().reset_index())

## Selection stability across earlier-season fits

Retention frequency measures whether a signal survives different training histories, not whether it is statistically significant. This table uses the full candidate block only. All underlying per-fit rejection reasons remain in the durable artifact archive.

In [ ]:
stability = pd.read_csv(RESULTS / "selection_stability.csv").merge(registry[["feature", "family"]], on="feature", validate="many_to_one")
table(stability.sort_values(["retention_rate", "retained_folds"], ascending=False).groupby(["Gender", "model"], sort=False).head(8))

## Does a feature family improve Brier score?

The official metric formula is game-weighted Brier, mean squared probability error. Mean-season Brier is the prespecified selection criterion and is reported separately. Whole physical tournament games are scored once; mirrored rows occur only in training. These five development seasons and the later consumed benchmark are retrospective, not an untouched holdout.

In [ ]:
scores = pd.read_csv(RESULTS / "leaderboard.csv")
table(scores.sort_values(["Gender", "macro_season_brier"]).groupby(["Gender", "model"], sort=False).head(6)[["Gender", "model", "block", "games", "brier", "macro_season_brier", "log_loss", "roc_auc"]])
# Fixed, question-driven comparison; the complete leaderboard remains in the report.
comparison_blocks = {
    "strength": "Strength baseline",
    "baseline_124": "Original bank",
    "full": "Expanded bank",
    "expanded_non_massey": "Expanded without Massey",
    "expanded_non_massey_no_coach": "Expanded without Massey/coaches",
    "expanded_non_massey_no_target": "Expanded without Massey/encoding",
    "rankings": "Strength + rankings",
    "coach_history": "Strength + coach history",
    "conference": "Strength + conference",
    "target_team": "Strength + team encoding",
    "target_seed": "Strength + seed encoding",
    "target_rank": "Strength + rank encoding",
    "dynamic": "Strength + dynamic ratings",
    "four_factors": "Strength + four factors",
}
for gender, label in [("M", "Men"), ("W", "Women")]:
    shown = scores.query("Gender == @gender and model == 'logistic'")
    shown = shown.loc[shown.block.isin(comparison_blocks)].copy()
    shown["Feature group"] = shown.block.map(comparison_blocks)
    bar_view(shown[["Feature group", "block", "brier", "macro_season_brier"]],
             "Feature group", "macro_season_brier", label + " · feature hypotheses under one logistic recipe")
display(Markdown("Focused comparison of the requested feature hypotheses. The complete block "
                 "leaderboard and histogram-boosting results remain in the recorded CSV evidence."))


## Paired ablations and uncertainty

Negative Brier differences favor the candidate. Intervals resample complete seasons and pair the same games; with only five development seasons and many comparisons they are exploratory, not corrected significance claims. The explicit `without_massey` comparison removes all 23 ranking-derived inputs together, rather than leaving trend or target-ranking derivatives behind.

In [ ]:
intervals = pd.read_csv(RESULTS / "ablation_intervals.csv")
new_families = ["distribution", "venue_profile", "opponent_profile", "trajectory", "peer_profile", "coach_history", "conference"]
selected = intervals.loc[(intervals.candidate.isin(new_families) & intervals.baseline.eq("strength")) | intervals.baseline.eq("without_massey")]
table(selected[["Gender", "model", "candidate", "baseline", "brier_delta", "ci_low", "ci_high", "season_count"]])
assert selected.season_count.eq(5).all()
controlled = intervals.loc[intervals.candidate.isin(["full", "expanded_non_massey"]) & intervals.baseline.str.startswith(("baseline_124", "expanded_non_massey"))]
table(controlled[["Gender", "model", "candidate", "baseline", "brier_delta", "ci_low", "ci_high", "season_count"]])


## Are 128 retained features enough?

The 3,106-column bank is a search space. The original 128-input limit was a fixed regularization choice, not an established optimum. This follow-up varies only the retention limit: **32, 64, 128 and 256**, holding the candidate matrix, redundancy threshold, and logistic/histogram recipes fixed. Men are compared with and without Massey; women use the available non-Massey sources.

Every fit screens its own earlier training games. A separate **nested** stream selects the limit by mean-season Brier on strictly earlier OOF seasons; ties favor fewer inputs. We retain all fixed-limit results, including unsuccessful ones. Existing 128-input outer fits are restored from verified checkpoints; cloning their estimator templates discards fitted state before any new training.

This is an exploratory capacity sensitivity study on previously explored development years. It does not make the 2022–2025 benchmark fresh, and it does not automatically replace notebook 03's model selection or notebook 04's frozen recipe.

In [ ]:
from march_mania.publication import capacity, followups
if MODE == "train":
    followups.study_stage(ROOT, "feature_capacity")
CAPACITY_DIR, CAPACITY_RECORD = capacity.review(ROOT)
capacity_summary = CAPACITY_RECORD["summary"]
display(Markdown(
    f"**Completed capacity study:** `{capacity_summary['fingerprint']}`  \n"
    f"**New fits:** {capacity_summary['new_fits']} · "
    f"**Verified existing fits reused:** {capacity_summary['inherited_reference_fits']}"
))
capacity_scores = pd.read_csv(CAPACITY_DIR / "leaderboard.csv")
capacity_choices = pd.read_csv(CAPACITY_DIR / "selection.csv")
assert (capacity_choices.history_last_season < capacity_choices.Season).all()
assert (capacity_choices.history_seasons >= 2).all()
labels = {"M_rankings": "Men · with Massey", "M_common": "Men · without Massey", "W_common": "Women"}
capacity_fixed = capacity_scores.query("block != 'nested'").copy()
capacity_fixed["retained_limit"] = capacity_fixed.block.str.removeprefix("cap_").astype(int)
capacity_fixed["population"] = capacity_fixed.route.map(labels)
capacity_fixed = capacity_fixed.sort_values(["population", "model", "retained_limit"])
fig = px.line(capacity_fixed, x="retained_limit", y="macro_season_brier", color="model",
              facet_col="population", markers=True, log_x=True,
              labels={"retained_limit": "Retained feature limit", "macro_season_brier": "Mean season Brier"},
              title="Does retaining more candidates improve Brier score?", height=440)
fig.update_layout(template="plotly_white", margin=dict(l=45, r=25, t=90, b=60))
fig.update_xaxes(tickvals=[32, 64, 128, 256])
fig.update_yaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
static, axes = plt.subplots(1, 3, figsize=(14, 4.6), constrained_layout=True)
for ax, (route, label) in zip(axes, labels.items()):
    for model, color in [("hist", "#526b9a"), ("logistic", "#147d82")]:
        panel = capacity_fixed.query("route == @route and model == @model")
        ax.plot(panel.retained_limit, panel.macro_season_brier, marker="o", color=color, label=model)
    ax.set_xscale("log", base=2)
    ax.set_xticks([32, 64, 128, 256], labels=[32, 64, 128, 256])
    ax.set(title=label, xlabel="Retained feature limit", ylabel="Mean season Brier · lower is better")
    ax.grid(alpha=0.2)
    ax.legend(frameon=False)
static.suptitle("Fixed recipes and identical season splits", fontsize=14)
buffer = io.BytesIO()
static.savefig(buffer, format="png", dpi=140)
plt.close(static)
bundle = {"application/vnd.plotly.v1+json": json.loads(fig.to_json()),
          "image/png": base64.b64encode(buffer.getvalue()).decode(),
          "text/plain": "Feature capacity sensitivity across the same five development seasons"}
display(bundle, raw=True)
table(capacity_scores.pivot(index=["route", "model"], columns="block", values="macro_season_brier").reset_index())
capacity_intervals = pd.read_csv(CAPACITY_DIR / "intervals.csv")
table(capacity_intervals.query("candidate == 'nested'")[["route", "model", "brier_delta", "ci_low", "ci_high"]])


**Measured conclusion.** Forward-selected capacity changes logistic mean-season Brier from 0.207742 to 0.192095 for men with Massey, from 0.204492 to 0.194492 without Massey, and from 0.171561 to 0.164711 for women. All six nested comparisons' season-bootstrap intervals include zero. The smaller broad models still trail the compact leaders in notebook 03.

Retaining 256 inputs generally hurts. This supports stopping blind expansion of the retained feature count. It does not establish 128 as universally optimal, exhaust every possible nonlinear representation, or demonstrate a new forecasting winner. The main model and benchmark recipes retain their original identities; no consumed benchmark was used to select a replacement.

## Does the consensus discard useful ranking information?

A further hypothesis uses **105 ranking systems known before 2013**, generating **840 additional candidates**: individual levels, nonlinear tail separation, disagreement with consensus, 7/30/60-day movement and availability. The system vocabulary is frozen from legal 2003–2012 snapshots; later systems cannot enter retrospectively. All values retain the existing publication-cohort and staleness rules.

The compact base is preserved. Each addition gets at most **16 training-screened inputs**; the embedding arm uses **eight training-fitted PCA components** from system levels and deviations. The same two fixed estimators and physical-game splits isolate representation changes. Separate strength and consensus controls keep the no-Massey comparison explicit. The [declared protocol](../docs/ranking_systems.md) gives each hypothesis, availability boundary and stopping question.

The figures below report **retrospective feature research**, with paired season-bootstrap intervals. They do not supply a fresh holdout or replace notebook 03/04's frozen recipe.

In [ ]:
from march_mania.publication import ranking_systems
if MODE == "train":
    followups.study_stage(ROOT, "ranking_systems")
SYSTEM_DIR, SYSTEM_RECORD = ranking_systems.review(ROOT)
system_summary = SYSTEM_RECORD["summary"]
system_scores = pd.read_csv(SYSTEM_DIR / "leaderboard.csv")
system_intervals = pd.read_csv(SYSTEM_DIR / "intervals.csv")
system_choices = pd.read_csv(SYSTEM_DIR / "selection.csv")
assert (system_choices.history_last_season < system_choices.Season).all()
assert system_summary["additional_candidates"] == 840
assert not system_summary["benchmark_labels_used"]
display(Markdown(
    f"**Verified system study:** `{system_summary['fingerprint']}`  \n"
    f"**Candidates:** {system_summary['additional_candidates']} additional · "
    f"**Evaluated fits:** {system_summary['fit_tasks']} · "
    f"**Original controls reused:** {system_summary['inherited_reference_fits']}"
))
table(system_scores[["model", "block", "macro_season_brier", "brier", "log_loss", "roc_auc"]])
contrasts = system_intervals.copy()
contrasts["plus"] = contrasts.ci_high - contrasts.brier_delta
contrasts["minus"] = contrasts.brier_delta - contrasts.ci_low
fig = px.scatter(contrasts, x="brier_delta", y="candidate", facet_col="model",
                 error_x="plus", error_x_minus="minus", color="model",
                 title="Does individual-system information beat the compact consensus?",
                 labels={"brier_delta": "Mean season Brier change · negative favors addition", "candidate": "Representation"},
                 height=490)
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(template="plotly_white", showlegend=False, margin=dict(l=100, r=35, t=80, b=70))
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
static, axes = plt.subplots(1, 2, figsize=(12, 4.7), constrained_layout=True)
for ax, model in zip(axes, ["hist", "logistic"]):
    panel = contrasts.query("model == @model")
    ax.errorbar(panel.brier_delta, panel.candidate, xerr=[panel.minus, panel.plus], fmt="o", color="#147d82", capsize=3)
    ax.axvline(0, linestyle="--", color="gray", linewidth=1)
    ax.set(title=model, xlabel="Brier change vs consensus · negative is better")
    ax.grid(axis="x", alpha=.2)
static.suptitle("Individual-system additions · exploratory 95% season intervals")
buffer = io.BytesIO()
static.savefig(buffer, format="png", dpi=140)
plt.close(static)
display({"application/vnd.plotly.v1+json": json.loads(fig.to_json()), "image/png": base64.b64encode(buffer.getvalue()).decode()}, raw=True)


### Does the effect survive across seasons?

A mean improvement can conceal a single favorable tournament. Negative cells favor the addition over the **same estimator's consensus control**. These are diagnostics of the declared arms; the forward-selected stream only uses earlier OOF losses. Retention frequencies count separate training fits, not a globally selected feature set. PCA input columns are labeled *embedded*, not counted as hundreds of fitted dimensions.

In [ ]:
system_years = pd.read_csv(SYSTEM_DIR / "metrics_by_season.csv")
reference = system_years.query("block == 'consensus'")[["model", "Season", "brier"]].rename(columns={"brier": "reference_brier"})
by_year = system_years.merge(reference, on=["model", "Season"], validate="many_to_one")
by_year["brier_change"] = by_year.brier - by_year.reference_brier
by_year = by_year.query("block != 'consensus'").copy()
by_year["label"] = by_year.model + " · " + by_year.block
heat = by_year.pivot(index="label", columns="Season", values="brier_change")
limit = float(abs(heat).to_numpy().max())
fig = px.imshow(heat, color_continuous_scale="RdBu_r", zmin=-limit, zmax=limit,
                text_auto=".3f", aspect="auto", title="Which seasons support the feature effect?",
                labels=dict(color="Brier change"), height=640)
fig.update_layout(template="plotly_white", margin=dict(l=180, r=35, t=65, b=50))
static, ax = plt.subplots(figsize=(10.5, 6.7), constrained_layout=True)
mat = ax.imshow(heat.to_numpy(), cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
ax.set_xticks(range(len(heat.columns)), labels=heat.columns.astype(str))
ax.set_yticks(range(len(heat)), labels=heat.index)
for i, row in enumerate(heat.to_numpy()):
    for j, value in enumerate(row):
        ax.text(j, i, f"{value:+.3f}", ha="center", va="center", fontsize=8,
                color="white" if abs(value) > limit * .6 else "black")
ax.set_title("Brier change vs the same estimator's consensus control")
static.colorbar(mat, ax=ax, label="Negative favors addition")
buffer = io.BytesIO()
static.savefig(buffer, format="png", dpi=140)
plt.close(static)
display({"application/vnd.plotly.v1+json": json.loads(fig.to_json()), "image/png": base64.b64encode(buffer.getvalue()).decode()}, raw=True)
table(system_choices)
system_screen = pd.read_csv(SYSTEM_DIR / "screening_summary.csv")
table(system_screen.groupby(["scope", "status"], as_index=False)["count"].sum())
system_stability = pd.read_csv(SYSTEM_DIR / "stability.csv")
table(system_stability.query("block == 'combined' and feature.str.startswith('diff_system_')", engine="python").sort_values(["model", "seasons", "feature"], ascending=[True, False, True]).groupby("model").head(12))


**Measured decision.** Individual levels improve fixed logistic mean-season Brier by only **0.000684**, with a 95% season-bootstrap interval spanning harm and benefit. The combined arm selects the same 16 additions and produces identical forecasts. Its 22 distinct retained columns across outer fits are all system-specific logits; 818 of 840 additions never survive the combined outer screens. Deviations, momentum, availability and the PCA embedding fail to improve the mean result.

The forward-selected logistic representation **worsens** Brier by **0.007632**; its exploratory interval excludes zero. Histogram selection mostly returns to the compact consensus. Thus, the small winning fixed-arm score does not justify promotion. The main model recipe remains frozen. This experiment, the broad family ablations and the capacity study jointly support diminishing returns from further variations of these official inputs. A useful new data source would constitute a different hypothesis, requiring verified historical availability and new validation evidence.

## Durability and interpretation

Snapshots are checkpointed by season and population. Submission features are generated in bounded 512-pair chunks; completed chunks are checksum-verified on resume. Every estimator, selection decision and prediction task has source/data/runtime fingerprints, UTC progress events and heartbeats. A changed feature definition cannot silently reuse old scores. Notebook 03 retrains against this exact feature fingerprint; notebook 05 compares matched old/new predictions. The largest candidate block is not automatically the winning model.